# Generate synthetic ECGs (balanced-sex–trained SSSD)

This notebook generates **female-only** synthetic ECGs using the **balanced-sex generator** at **checkpoint `20000.pkl`**.

Design choices (as requested):
- Base target size: `N_base = 5%` of **`df_train_real`** size.
- Stratification: exact **unique 71-d SCP multihot patterns** among **female train rows**.
- Allocation: proportional to female train distribution (largest-remainder integer allocation).
- MI upweighting: if a pattern contains MI, generate **3x** that pattern count.
- Therefore total generated size is larger than `N_base`.

The notebook saves:
- combined arrays: `syn_data_12.npy`, `syn_cond_72.npy`
- metadata: `syn_meta.json`, `group_specs_final.json`
- optional per-pattern shards (toggle).

In [ ]:
!pip install -q pykeops
!pip install -q wfdb resampy ishneholterlib pytorch-lightning

In [ ]:
import os
import sys
from pathlib import Path

REPO_URL = "https://github.com/Anonymous0002396/CMED-Mini-Project.git"
PROJECT_ROOT = Path("/content/CMED-Mini-Project")
REPO_ROOT = PROJECT_ROOT / "SSSD-ECG"

if not PROJECT_ROOT.exists():
    !git clone {REPO_URL} /content/CMED-Mini-Project
else:
    %cd /content/CMED-Mini-Project
    !git pull

sys.path.insert(0, str(REPO_ROOT / "src"))
sys.path.insert(0, str(REPO_ROOT / "src" / "ptb_xl"))
sys.path.insert(0, str(REPO_ROOT / "src" / "sssd"))

print("REPO_ROOT:", REPO_ROOT)
print("train.py exists:", (REPO_ROOT / "src" / "sssd" / "train.py").exists())

In [ ]:
from google.colab import drive
from pathlib import Path

drive.mount('/content/drive')

if not os.path.exists('/content/ptbxl.zip'):
    !cp "/content/drive/MyDrive/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1.zip" /content/ptbxl.zip

if not os.path.exists('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1'):
    !unzip -oq /content/ptbxl.zip -d /content/
    print("Extraction complete.")
else:
    print("PTB-XL already extracted, skipping extraction.")

raw_ptbxl = Path('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1')
print('raw_ptbxl exists:', raw_ptbxl.exists())

In [ ]:
import numpy as np
import pandas as pd
from collections import defaultdict
from pathlib import Path

from clinical_ts.ecg_utils import prepare_data_ptb_xl, channel_stoi_default
from clinical_ts.timeseries_utils import reformat_as_memmap, load_dataset

# Rebuild processed PTB-XL at 100 Hz (same preprocessing family as training notebook)
target_fs = 100
data_folder_ptb_xl = Path('/content/ptb-xl-a-large-publicly-available-electrocardiography-dataset-1')
target_folder_ptb_xl = Path(f'/content/processed_ptb_xl_fs{target_fs}')

if target_folder_ptb_xl.exists():
    !rm -rf {target_folder_ptb_xl}

print('Rebuilding processed PTB-XL at:', target_folder_ptb_xl)

df_ptb_xl, lbl_itos_ptb_xl, mean_ptb_xl, std_ptb_xl = prepare_data_ptb_xl(
    data_path=data_folder_ptb_xl,
    min_cnt=0,
    target_fs=target_fs,
    channels=12,
    channel_stoi=channel_stoi_default,
    target_folder=target_folder_ptb_xl,
    recreate_data=True,
)

reformat_as_memmap(
    df_ptb_xl,
    target_folder_ptb_xl / 'memmap.npy',
    data_folder=target_folder_ptb_xl,
    delete_npys=True,
)

df_mapped, lbl_itos_dict, mean, std = load_dataset(target_folder_ptb_xl)
print('Processed df shape:', df_mapped.shape)

# Build 71-d multihot + MI diagnostic + sex
label_key = 'label_all'
label_names = np.array(lbl_itos_dict[label_key])
assert len(label_names) == 71, len(label_names)

mi_statement_names = [
    'IMI', 'ASMI', 'ILMI', 'AMI', 'ALMI',
    'INJAS', 'LMI', 'INJAL', 'IPLMI', 'IPMI',
    'INJIN', 'PMI', 'INJLA', 'INJIL'
]
label_name_to_idx = {str(name): i for i, name in enumerate(label_names)}
mi_label_indices = [label_name_to_idx[name] for name in mi_statement_names]

def multihot_encode(indices, num_classes):
    out = np.zeros(num_classes, dtype=np.float32)
    for i in indices:
        out[i] = 1.0
    return out

def row_multihot_to_binary_mi(row_vec, mi_indices):
    return float(row_vec[mi_indices].sum() > 0)

df_real = df_mapped.copy()
df_real['label_multihot'] = df_real[f'{label_key}_numeric'].apply(lambda x: multihot_encode(x, len(label_names)))
df_real['label_mi'] = df_real['label_multihot'].apply(lambda x: row_multihot_to_binary_mi(x, mi_label_indices)).astype(np.float32)
df_real['sex_binary'] = df_real['sex'].astype(np.float32)

# Train split only (as requested)
max_fold_id = int(df_real['strat_fold'].max())
df_train_real = df_real[df_real['strat_fold'] < max_fold_id - 1].copy()

print('Train split size:', len(df_train_real))
print('Female train size:', int((df_train_real['sex_binary'] == 1.0).sum()))
print('Train counts by (sex, MI):')
print(df_train_real.groupby(['sex_binary', 'label_mi']).size())

In [ ]:
# Female-only target construction:
# N_base = 5% of df_train_real, then pattern-proportional allocation among female rows,
# then 3x upweight for MI-containing patterns.
import hashlib

BASE_FRAC = 0.05
N_BASE = max(1, int(round(BASE_FRAC * len(df_train_real))))
print('N_BASE (5% of train split):', N_BASE)

df_train_female = df_train_real[df_train_real['sex_binary'] == 1.0].copy()
if len(df_train_female) == 0:
    raise RuntimeError('No female rows found in df_train_real')

print('Female pool size:', len(df_train_female))

def _largest_remainder_allocation(n_target, counts_dict):
    keys = list(counts_dict.keys())
    total = sum(counts_dict[k] for k in keys)
    if total <= 0:
        raise ValueError('Invalid counts_dict total')
    ideal = {k: n_target * (counts_dict[k] / total) for k in keys}
    floors = {k: int(np.floor(ideal[k])) for k in keys}
    rem = n_target - sum(floors.values())
    order = sorted(keys, key=lambda k: (ideal[k] - floors[k]), reverse=True)
    for i in range(rem):
        floors[order[i]] += 1
    return floors

# Build female strata by exact 71-d pattern
female_pools = defaultdict(list)   # key bytes -> row indices
pattern_vec = {}                   # key bytes -> np.array(71,)

for idx, row in df_train_female.iterrows():
    mh = np.asarray(row['label_multihot'], dtype=np.float32)
    k = mh.tobytes()
    female_pools[k].append(int(idx))
    pattern_vec[k] = mh

female_counts = {k: len(v) for k, v in female_pools.items()}
base_targets = _largest_remainder_allocation(N_BASE, female_counts)

# Keep only strata with nonzero base target
active_keys = [k for k, n in base_targets.items() if n > 0]

# 3x rule for MI-containing patterns
final_specs = []
for j, k in enumerate(active_keys):
    mh = pattern_vec[k]
    base_n = int(base_targets[k])
    has_mi = bool(mh[mi_label_indices].sum() > 0.0)
    final_n = int(base_n * (3 if has_mi else 1))

    # deterministic compact id from pattern bytes
    pattern_id = hashlib.md5(k).hexdigest()[:10]

    cond72 = np.concatenate([[1.0], mh.astype(np.float32)], axis=0).astype(np.float32)
    final_specs.append(
        {
            'name': f'female_pattern_{j:04d}_{pattern_id}' + ('_mi' if has_mi else '_nonmi'),
            'pattern_id': pattern_id,
            'base_n': base_n,
            'n': final_n,
            'is_mi_pattern': has_mi,
            'seed': int(7000 + j),
            'cond': cond72,
        }
    )

N_FINAL = int(sum(g['n'] for g in final_specs))
N_MI = int(sum(g['n'] for g in final_specs if g['is_mi_pattern']))
N_NONMI = int(sum(g['n'] for g in final_specs if not g['is_mi_pattern']))

print('Active female patterns:', len(final_specs))
print('Total generated N after MI x3 upweight:', N_FINAL)
print('  MI-pattern generated count:', N_MI)
print('  non-MI-pattern generated count:', N_NONMI)
print('  MI share:', N_MI / max(1, N_FINAL))

# sanity checks
assert all(spec['n'] >= 0 for spec in final_specs)
assert all(np.asarray(spec['cond']).shape[0] == 72 for spec in final_specs)

# Preview first few strata
for spec in final_specs[:10]:
    print(spec['name'], '| base_n=', spec['base_n'], '-> n=', spec['n'], '| MI=', spec['is_mi_pattern'])

In [ ]:
import json
import torch
from pathlib import Path

from models.SSSD_ECG import SSSD_ECG
from utils.util import calc_diffusion_hyperparams

%cd /content/CMED-Mini-Project/SSSD-ECG/src/sssd

# Balanced-sex config (write fallback if missing)
config_dir = Path('config')
config_dir.mkdir(parents=True, exist_ok=True)
cfg_path = config_dir / 'config_SSSD_ECG_sex_scp71_balanced_50f_50m.json'

DRIVE_OUTPUT_DIR_FOR_CFG = '/content/drive/MyDrive/sssd_sex_scp71_balanced_50f_50m'

if not cfg_path.is_file():
    default_cfg = {
        'diffusion_config': {'T': 200, 'beta_0': 0.0001, 'beta_T': 0.02},
        'wavenet_config': {
            'in_channels': 8,
            'out_channels': 8,
            'num_res_layers': 36,
            'res_channels': 256,
            'skip_channels': 256,
            'diffusion_step_embed_dim_in': 128,
            'diffusion_step_embed_dim_mid': 512,
            'diffusion_step_embed_dim_out': 512,
            's4_lmax': 1000,
            's4_d_state': 64,
            's4_dropout': 0.0,
            's4_bidirectional': 1,
            's4_layernorm': 1,
            'label_embed_dim': 128,
            'label_embed_classes': 72,
        },
        'train_config': {
            'output_directory': DRIVE_OUTPUT_DIR_FOR_CFG,
            'ckpt_iter': 'max',
            'iters_per_ckpt': 2000,
            'iters_per_logging': 1000,
            'n_iters': 20000,
            'learning_rate': 2e-4,
            'batch_size': 8,
        },
        'trainset_config': {
            'segment_length': 1000,
            'sampling_rate': 100,
            'finetune_dataset': 'ptbxl_sex_scp71_balanced_50f_50m',
        },
        'gen_config': {
            'output_directory': DRIVE_OUTPUT_DIR_FOR_CFG,
            'ckpt_path': DRIVE_OUTPUT_DIR_FOR_CFG,
        },
    }
    cfg_path.write_text(json.dumps(default_cfg, indent=2))
    print('Wrote fallback config:', cfg_path.resolve())

cfg = json.loads(cfg_path.read_text())
diffusion_config = cfg['diffusion_config']
model_config = cfg['wavenet_config']
train_config = cfg['train_config']

out_root = Path(train_config['output_directory'])
local_path = 'ch{}_T{}_betaT{}'.format(
    model_config['res_channels'],
    diffusion_config['T'],
    diffusion_config['beta_T'],
)
ckpt_dir = out_root / local_path
ckpt_iter = 20000
ckpt_path = ckpt_dir / f'{ckpt_iter}.pkl'

# Strict requirement: checkpoint 20000 must exist
if not ckpt_path.exists():
    raise FileNotFoundError(
        f'Required checkpoint missing: {ckpt_path}. '
        'Please train until 20000 and re-run.'
    )

print('Using checkpoint:', ckpt_path)

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('device:', device)

net = SSSD_ECG(**model_config).to(device)
checkpoint = torch.load(ckpt_path, map_location='cpu')
net.load_state_dict(checkpoint['model_state_dict'])
net.eval()

diffusion_hyperparams = calc_diffusion_hyperparams(**diffusion_config)
for k in diffusion_hyperparams:
    if k != 'T':
        diffusion_hyperparams[k] = diffusion_hyperparams[k].to(device)

print('Model loaded at iteration', ckpt_iter)

In [ ]:
import json
import numpy as np
import torch
from pathlib import Path
from numpy.lib.format import open_memmap

from utils.util import sampling_label
from inference import generate_four_leads

# Save location
save_dir = Path('/content/drive/MyDrive/sex_scp71_synthetic_balanced_female_mi3x_ckpt20000')
save_dir.mkdir(parents=True, exist_ok=True)

groups_dir = save_dir / "groups"
groups_dir.mkdir(parents=True, exist_ok=True)

batch_size_gen = 16
SAVE_EVERY_BATCHES = 10  # writes progress every N batches

# ---------- per-group generation with resume ----------
for i, spec in enumerate(final_specs):
    name = spec['name']
    cond_vec = np.asarray(spec['cond'], dtype=np.float32)
    n = int(spec['n'])
    seed = int(spec['seed'])

    if n <= 0:
        continue

    grp_data_path = groups_dir / f"{name}_data.npy"
    grp_cond_path = groups_dir / f"{name}_cond.npy"
    grp_state_path = groups_dir / f"{name}_state.json"
    grp_meta_path = groups_dir / f"{name}_meta.json"

    # load previous progress if exists
    done = 0
    if grp_state_path.exists():
        st = json.loads(grp_state_path.read_text())
        done = int(st.get("done", 0))

    # completed already
    if done >= n and grp_data_path.exists() and grp_cond_path.exists():
        print(f"[{i+1}/{len(final_specs)}] Skip completed {name} ({done}/{n})")
        continue

    print(f"[{i+1}/{len(final_specs)}] Generating {name} | MI={spec['is_mi_pattern']} | resume {done}/{n}")

    # deterministic generation for this group
    torch.manual_seed(seed)
    np.random.seed(seed)

    # create/open memmaps
    if grp_data_path.exists() and grp_cond_path.exists():
        data_mm = open_memmap(grp_data_path, mode='r+')
        cond_mm = open_memmap(grp_cond_path, mode='r+')
        # sanity shape checks
        assert data_mm.shape == (n, 12, 1000), (name, data_mm.shape, n)
        assert cond_mm.shape == (n, 72), (name, cond_mm.shape, n)
    else:
        data_mm = open_memmap(grp_data_path, mode='w+', dtype=np.float32, shape=(n, 12, 1000))
        cond_mm = open_memmap(grp_cond_path, mode='w+', dtype=np.float32, shape=(n, 72))

    batch_idx = 0
    while done < n:
        b = min(batch_size_gen, n - done)

        cond_batch_np = np.repeat(cond_vec[None, :], b, axis=0).astype(np.float32)
        cond_batch = torch.tensor(cond_batch_np, dtype=torch.float32, device=device)

        with torch.no_grad():
            gen8 = sampling_label(
                net,
                (b, 8, 1000),
                diffusion_hyperparams,
                cond=cond_batch,
            )
            gen12 = generate_four_leads(gen8)

        gen12_np = gen12.detach().cpu().numpy().astype(np.float32)

        # write directly into memmap
        data_mm[done:done+b] = gen12_np
        cond_mm[done:done+b] = cond_batch_np
        done += b
        batch_idx += 1

        # periodic flush + checkpoint
        if (batch_idx % SAVE_EVERY_BATCHES == 0) or (done == n):
            data_mm.flush()
            cond_mm.flush()
            grp_state_path.write_text(json.dumps({
                "name": name,
                "done": int(done),
                "n": int(n),
                "seed": int(seed),
                "checkpoint_iter": int(ckpt_iter),
            }, indent=2))
            print(f"   saved progress {done}/{n}")

    # group finished metadata
    grp_meta_path.write_text(json.dumps({
        "name": name,
        "pattern_id": spec['pattern_id'],
        "is_mi_pattern": bool(spec['is_mi_pattern']),
        "base_n": int(spec['base_n']),
        "n": int(n),
        "seed": int(seed),
        "checkpoint_iter": int(ckpt_iter),
        "checkpoint_path": str(ckpt_path),
        "batch_size_gen": int(batch_size_gen),
        "completed": True
    }, indent=2))

print("\nPer-group generation finished/resumed.")

# ---------- combine from group files (also resumable-ish by rerun) ----------
total_n = int(sum(int(g['n']) for g in final_specs if int(g['n']) > 0))
syn_data_path = save_dir / "syn_data_12.npy"
syn_cond_path = save_dir / "syn_cond_72.npy"
syn_names_path = save_dir / "syn_group_names.npy"

syn_data_mm = open_memmap(syn_data_path, mode='w+', dtype=np.float32, shape=(total_n, 12, 1000))
syn_cond_mm = open_memmap(syn_cond_path, mode='w+', dtype=np.float32, shape=(total_n, 72))
all_names = []

cursor = 0
for spec in final_specs:
    name = spec['name']
    n = int(spec['n'])
    if n <= 0:
        continue

    grp_data_path = groups_dir / f"{name}_data.npy"
    grp_cond_path = groups_dir / f"{name}_cond.npy"
    grp_state_path = groups_dir / f"{name}_state.json"

    if (not grp_data_path.exists()) or (not grp_cond_path.exists()) or (not grp_state_path.exists()):
        raise FileNotFoundError(f"Missing group outputs for {name}")

    st = json.loads(grp_state_path.read_text())
    if int(st.get("done", 0)) < n:
        raise RuntimeError(f"Group {name} not complete: {st.get('done',0)}/{n}")

    gd = np.load(grp_data_path, mmap_mode='r')
    gc = np.load(grp_cond_path, mmap_mode='r')

    syn_data_mm[cursor:cursor+n] = gd
    syn_cond_mm[cursor:cursor+n] = gc
    all_names.extend([name] * n)
    cursor += n

syn_data_mm.flush()
syn_cond_mm.flush()
np.save(syn_names_path, np.array(all_names, dtype=object))

with open(save_dir / 'group_specs_final.json', 'w') as f:
    json.dump(
        [
            {
                'name': g['name'],
                'pattern_id': g['pattern_id'],
                'base_n': int(g['base_n']),
                'n': int(g['n']),
                'is_mi_pattern': bool(g['is_mi_pattern']),
                'seed': int(g['seed']),
            }
            for g in final_specs
        ],
        f,
        indent=2,
    )

with open(save_dir / 'syn_meta.json', 'w') as f:
    json.dump(
        {
            'design': 'female-only; base 5% of df_train_real; exact 71-pattern strata; MI patterns x3',
            'base_fraction': float(BASE_FRAC),
            'N_base': int(N_BASE),
            'N_final_generated': int(total_n),
            'num_patterns_active': int(len(final_specs)),
            'checkpoint_iter': int(ckpt_iter),
            'checkpoint_path': str(ckpt_path),
            'groups_dir': str(groups_dir),
            'save_dir': str(save_dir),
        },
        f,
        indent=2,
    )

print("\nDone.")
print("syn_data_12:", np.load(syn_data_path, mmap_mode='r').shape)
print("syn_cond_72:", np.load(syn_cond_path, mmap_mode='r').shape)
print("saved under:", save_dir)

In [ ]:
print(4)